# **Dados e Aprendizagem Automática** 

## **XGBoost - Tratamento 2 com Grid Search**
Nesta abordagem, partindo do tratamento 2, mantivemos a engenharia de atributos temporais e a normalização de texto. Para a modelação aplicamos à Grid Search hiperparâmetros chave (n_estimators, max_depth, learning_rate, subsample, colsample_bytree, min_child_weight, gamma) para equilibrar desempenho e tempo de treino. O alvo é codificado com LabelEncoder (inteiros) para compatibilidade com XGBoost, e as previsões são descodificadas para etiquetas originais.

**Imports necessários para este teste**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

%matplotlib inline

### **Preparation**
**Load the CSVs**

In [2]:
df_train = pd.read_csv('../../Datasets/training_data.csv', encoding='latin-1')
df_test = pd.read_csv('../../Datasets/test_data.csv', encoding='latin-1')

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

Train shape: (6812, 14)
Test shape: (1500, 13)


**Feature Engineering (Extração da data)**

In [3]:
def extract_date_features(df):
    df['record_date'] = pd.to_datetime(df['record_date'])
    df['hour'] = df['record_date'].dt.hour
    df['day_of_week'] = df['record_date'].dt.dayofweek # Monday=0, Sunday=6
    df['month'] = df['record_date'].dt.month
    
    # Create "Weekend" feature
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
    
    # Create "Rush Hour" feature (7 da manhã até às 9 da manhã e 4 da tarde ate às 7 da tarde, podemos brincar com estas horas)
    df['is_rush_hour'] = df['hour'].apply(lambda x: 1 if (7 <= x <= 9) or (16 <= x <= 19) else 0)
    
    return df.drop(columns=['record_date'])

df_train = extract_date_features(df_train)
df_test = extract_date_features(df_test)


**Missing Values e Valores Incorretos**

In [4]:
def clean_categorical_text(df):

    # Primeiro "limpamos" a coluna 'AVERAGE CLOUDINESS'
    correcoes_erros = {
        'cï¿½u': 'ceu',    # erro especifico
        'c\u00e9u': 'ceu', # é
        'c\u00fa': 'ceu',  # ú
        'c\u00fau': 'ceu', 
        'céu': 'ceu'
    }
    # regex=True permite substituir apenas parte da frase (ex: "cï¿½u claro" -> "ceu claro")
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].astype(str).replace(correcoes_erros, regex=True)

    cloud_map = {
        'ceu claro': 'ceu_claro',
        'ceu limpo': 'ceu_claro',

        'ceu pouco nublado': 'pouco_nublado',
        'nuvens dispersas': 'pouco_nublado',
        'algumas nuvens': 'pouco_nublado',

        'nuvens quebrados': 'nublado', 
        'nuvens quebradas': 'nublado',
        'tempo nublado': 'nublado',
        'nublado': 'nublado',
    }
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].astype(str).replace(cloud_map, regex=True)
    # Tratamos também dos seus missing values
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].replace('nan', 'unknown_cloudiness')
    
    # Depois "limpamos" também a coluna da "AVERAGE RAIN"
    rain_map = {
        'chuva fraca': 'chuva_fraca',
        'chuva leve': 'chuva_fraca',
        'aguaceiros fracos': 'chuva_fraca',
        'chuvisco fraco': 'chuva_fraca',
        'chuvisco e chuva fraca': 'chuva_fraca',
        'trovoada com chuva leve': 'chuva_fraca', 

        'chuva moderada': 'chuva_moderada',
        'chuva': 'chuva_moderada',
        'aguaceiros': 'chuva_moderada',
        'trovoada com chuva': 'chuva_moderada',

        'chuva forte': 'chuva_forte',
        'chuva de intensidade pesada': 'chuva_forte',
        'chuva de intensidade pesado': 'chuva_forte'
    }
    df['AVERAGE_RAIN'] = df['AVERAGE_RAIN'].replace(rain_map)
    # Tratamos também dos seus missing values
    df['AVERAGE_RAIN'] = df['AVERAGE_RAIN'].fillna('no_rain')
    
    #df["LUMINOSITY"] = df_train["LUMINOSITY"].replace("LOW_LIGHT", "LIGHT")
    
    return df

df_train = clean_categorical_text(df_train)
df_test = clean_categorical_text(df_test)

In [5]:
df_train["AVERAGE_SPEED_DIFF"] = df_train["AVERAGE_SPEED_DIFF"].fillna("None")

**Verificação dos valores dessas colunas agora**

In [6]:
df_test['AVERAGE_CLOUDINESS'].value_counts()

AVERAGE_CLOUDINESS
unknown_cloudiness    599
ceu_claro             372
pouco_nublado         301
nublado               228
Name: count, dtype: int64

In [7]:
df_test['AVERAGE_RAIN'].value_counts()

AVERAGE_RAIN
no_rain           1360
chuva_fraca         93
chuva_moderada      47
Name: count, dtype: int64

**Drop de Colunas Redundates** 

Considera-mo-las redundantes devido a 'CITY_NAME' conter um valor constante ("Porto") e 'AVERAGE_PRECIPITATION' consistir quase apenas em zeros.

In [8]:
cols_to_drop = ['city_name', 'AVERAGE_PRECIPITATION']
df_train = df_train.drop(columns=cols_to_drop)
df_test = df_test.drop(columns=cols_to_drop)

**Handling categoric data**

In [9]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OrdinalEncoder

In [10]:
"""
AVERAGE_RAIN
"""

le_rain = LabelEncoder()
df_train["AVERAGE_RAIN"] = le_rain.fit_transform(df_train["AVERAGE_RAIN"])
df_test["AVERAGE_RAIN"] = le_rain.transform(df_test["AVERAGE_RAIN"])


In [11]:
"""
AVERAGE_CLOUDINESS
"""
le_cloud = LabelEncoder()
df_train["AVERAGE_CLOUDINESS"] = le_cloud.fit_transform(df_train["AVERAGE_CLOUDINESS"])
df_test["AVERAGE_CLOUDINESS"] = le_cloud.transform(df_test["AVERAGE_CLOUDINESS"])


In [12]:
"""
LUMINOSITY
"""
le_lu = LabelEncoder()
df_train["LUMINOSITY"] = le_lu.fit_transform(df_train["LUMINOSITY"])
df_test["LUMINOSITY"] = le_lu.transform(df_test["LUMINOSITY"])


In [13]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6812 entries, 0 to 6811
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   AVERAGE_SPEED_DIFF       6812 non-null   object 
 1   AVERAGE_FREE_FLOW_SPEED  6812 non-null   float64
 2   AVERAGE_TIME_DIFF        6812 non-null   float64
 3   AVERAGE_FREE_FLOW_TIME   6812 non-null   float64
 4   LUMINOSITY               6812 non-null   int64  
 5   AVERAGE_TEMPERATURE      6812 non-null   float64
 6   AVERAGE_ATMOSP_PRESSURE  6812 non-null   float64
 7   AVERAGE_HUMIDITY         6812 non-null   float64
 8   AVERAGE_WIND_SPEED       6812 non-null   float64
 9   AVERAGE_CLOUDINESS       6812 non-null   int64  
 10  AVERAGE_RAIN             6812 non-null   int64  
 11  hour                     6812 non-null   int32  
 12  day_of_week              6812 non-null   int32  
 13  month                    6812 non-null   int32  
 14  is_weekend              

In [14]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   AVERAGE_FREE_FLOW_SPEED  1500 non-null   float64
 1   AVERAGE_TIME_DIFF        1500 non-null   float64
 2   AVERAGE_FREE_FLOW_TIME   1500 non-null   float64
 3   LUMINOSITY               1500 non-null   int64  
 4   AVERAGE_TEMPERATURE      1500 non-null   float64
 5   AVERAGE_ATMOSP_PRESSURE  1500 non-null   float64
 6   AVERAGE_HUMIDITY         1500 non-null   float64
 7   AVERAGE_WIND_SPEED       1500 non-null   float64
 8   AVERAGE_CLOUDINESS       1500 non-null   int64  
 9   AVERAGE_RAIN             1500 non-null   int64  
 10  hour                     1500 non-null   int32  
 11  day_of_week              1500 non-null   int32  
 12  month                    1500 non-null   int32  
 13  is_weekend               1500 non-null   int64  
 14  is_rush_hour            

### **Modeling**

Select features and target

In [15]:
X_train = df_train.drop(columns=["AVERAGE_SPEED_DIFF"])
y_train = df_train["AVERAGE_SPEED_DIFF"]

# Encode target for XGBoost
from sklearn.preprocessing import LabelEncoder
le_target = LabelEncoder()
y_train_enc = le_target.fit_transform(y_train)

In [16]:
X_test = df_test

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

**Grid Search**

Hyperparameter Tuning with Grid Search - Check what is the best estimator and parameters using pruning (ccp_alphas)

In [17]:
from sklearn.model_selection import GridSearchCV

In [18]:
# Initialize a baseline XGBoost model (fast 'hist' tree method)
base_model = XGBClassifier(random_state=752, tree_method='hist', eval_metric='mlogloss', n_jobs=-1)
base_model.fit(X_train, y_train_enc)
print("Baseline model trained. Default max_depth:", base_model.get_params().get('max_depth'))

Baseline model trained. Default max_depth: None


In [19]:
import numpy as np
n_classes = len(np.unique(y_train_enc))
print("Detected classes:", n_classes)

Detected classes: 5


In [20]:
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (6812, 15) Test shape: (1500, 15)


In [21]:
from sklearn.model_selection import GridSearchCV

# XGBoost hyperparameter grid simple
param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [5, 10, None],
    'learning_rate': [0.05, 0.1, 0.2],
    'min_child_weight': [1, 3],
    'gamma': [0, 1]
}

#(balanced for performance and time)
param_grid2 = {
    'n_estimators': [200, 400, 600],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3],
    'gamma': [0, 1]
}

xgb = XGBClassifier(random_state=752, tree_method='hist', eval_metric='mlogloss', n_jobs=-1)

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='accuracy',
    verbose=2
 )

grid.fit(X_train, y_train_enc)

print("Best Parameters:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)

# Best model
best_xgb = grid.best_estimator_

Fitting 3 folds for each of 72 candidates, totalling 216 fits


[CV] END gamma=0, learning_rate=0.05, max_depth=5, min_child_weight=1, n_estimators=100; total time=   0.9s
[CV] END gamma=0, learning_rate=0.05, max_depth=5, min_child_weight=3, n_estimators=100; total time=   0.9s
[CV] END gamma=0, learning_rate=0.05, max_depth=5, min_child_weight=1, n_estimators=100; total time=   1.1s
[CV] END gamma=0, learning_rate=0.05, max_depth=5, min_child_weight=3, n_estimators=100; total time=   1.3s
[CV] END gamma=0, learning_rate=0.05, max_depth=5, min_child_weight=1, n_estimators=100; total time=   1.3s
[CV] END gamma=0, learning_rate=0.05, max_depth=5, min_child_weight=3, n_estimators=100; total time=   1.0s
[CV] END gamma=0, learning_rate=0.05, max_depth=10, min_child_weight=1, n_estimators=100; total time=   2.4s
[CV] END gamma=0, learning_rate=0.05, max_depth=5, min_child_weight=3, n_estimators=300; total time=   2.4s
[CV] END gamma=0, learning_rate=0.05, max_depth=5, min_child_weight=3, n_estimators=300; total time=   2.4s
[CV] END gamma=0, learning_

**Validation Block**

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Copies
X_val_copy = X_train.copy()
y_val_copy = y_train_enc.copy()

# Validation split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_val_copy, y_val_copy, test_size=0.2, random_state=752, stratify=y_val_copy
)

# Validate with best XGBoost model from GridSearch
clf_val = best_xgb
clf_val.fit(X_tr, y_tr)

# Predict validation (encoded) and decode to original labels
val_preds = clf_val.predict(X_val)
val_preds_dec = le_target.inverse_transform(val_preds)
y_val_dec = le_target.inverse_transform(y_val)

print("Validation Accuracy:", accuracy_score(y_val_dec, val_preds_dec))
print("\nClassification Report:\n", classification_report(y_val_dec, val_preds_dec))

Validation Accuracy: 0.7997065297138665

Classification Report:
               precision    recall  f1-score   support

        High       0.76      0.77      0.77       213
         Low       0.69      0.72      0.70       284
      Medium       0.77      0.72      0.74       330
        None       0.89      0.91      0.90       440
   Very_High       0.90      0.86      0.88        96

    accuracy                           0.80      1363
   macro avg       0.80      0.80      0.80      1363
weighted avg       0.80      0.80      0.80      1363



**Predict**

Predict on test set

In [23]:
# Train best model on full training data (encoded labels)
clf = best_xgb
clf.fit(X_train, y_train_enc)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [24]:
preds = clf.predict(X_test)
# Decode predictions back to original labels
preds_dec = le_target.inverse_transform(preds)

Create csv

In [25]:
submission = pd.DataFrame({
    "RowId": range(1, len(preds_dec)+1),
    "Speed_Diff": preds_dec
})

submission.to_csv("../../results/XGBoost/xgboost2grid.csv", index=False)
submission.head()

,RowId,Speed_Diff
0,1,None
1,2,Low
2,3,None
3,4,High
4,5,Low
